In [1]:
from ultralytics import YOLO
import cv2
import numpy as np

In [2]:
model = YOLO('yolov8n.pt')

In [3]:
img_path = '../data/raw/8-2-first.jpg'
img = cv2.imread(img_path)

In [4]:
result = model(img)[0]


0: 480x640 13 cars, 41.8ms
Speed: 17.6ms preprocess, 41.8ms inference, 0.9ms postprocess per image at shape (1, 3, 480, 640)


In [5]:
#car, motorcycle, bus, truck
VALID_CLASSES = {2, 3, 5, 7}

vehicles = []
for box in result.boxes:
    cls = int(box.cls[0])
    if cls in VALID_CLASSES:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        vehicles.append((x1, y1, x2, y2, cls))


In [6]:
vehicles.sort(key= lambda b: b[0])

widths = [x2 - x1 for (x1, y1, x2, y2, cls) in vehicles]
avg_width = np.mean(widths)

In [7]:
THRESHOLD = 1.2 * avg_width

In [8]:
for (x1, y1, x2, y2, cls) in vehicles:
    cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 255), 2)

In [9]:
for i in range(len(vehicles) - 1):
        x1_a, y1_a, x2_a, y2_a, cls_a = vehicles[i]
        x1_b, y1_b, x2_b, y2_b, cls_b = vehicles[i + 1]

        gap = x1_b - x2_a

        if gap > THRESHOLD:
            # Draw the parking lot
            x_start = x2_a
            x_end = x1_b
            y_top = min(y1_a, y1_b)
            y_bottom = max(y2_a, y2_b)

            cv2.rectangle(img, (x_start, y_top), (x_end, y_bottom), (0, 255, 0), 2)
            cv2.putText(img, "Park Here", (x_start + 5, y_top - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

In [10]:
cv2.imwrite("../data/processed/resultado2.jpg", img)


True